# Lesson 19: How Everything Connects

You've been using the product to create articles. But what happens **behind the scenes** when you say "Create an article about SEO"?

This lesson traces the **full journey of a request** — from the moment you type a message to the moment an article appears on disk. No API calls — a guided tour of how the pieces fit together.

## The Journey of a Request

When you type "Create an article about on-page SEO" in the web interface:

```
You (browser)
  |
  v
serve.py --- AgentOS receives the request via SSE
  |
  v
agents/team.py --- Team Leader (Sonnet) decides who handles it
  |
  v
agents/content_writer.py --- Content Writer runs:
  |                            1. web_search() via DataForSEO
  |                            2. Writes the article
  |                            3. save_article() to disk
  |
  v
tools/storage.py --- Saves content/{id}.md + updates articles.json
  |
  v
content/ --- Article saved as a .md file
```

Each file has a **single responsibility**. One by one:

## serve.py — The Web Backend

`serve.py` is the **web entry point** — the file you run with `python output/backend/serve.py`.

It does three things:
1. Validates the `ANTHROPIC_API_KEY`
2. Wraps the team with **AgentOS** (auto-generates 50+ API endpoints)
3. Adds custom article routes (`/api/articles`)

AgentOS automatically creates endpoints like:
- `POST /teams/seo-workspace/runs` — send a prompt (supports SSE streaming)
- `GET /health` — health check
- `GET /docs` — Swagger API docs

The custom routes expose our storage layer:
- `GET /api/articles` — list all articles
- `GET /api/articles/{id}` — get one article
- `DELETE /api/articles/{id}` — delete an article

`chat.py` also exists as a CLI alternative — same team, different interface.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

# Let's look at serve.py
serve_path = os.path.abspath("../../output/backend/serve.py")
with open(serve_path, "r", encoding="utf-8") as f:
    serve_code = f.read()

print(f"serve.py ({len(serve_code.splitlines())} lines)\n")
for i, line in enumerate(serve_code.splitlines(), 1):
    print(f"  {i:3d} | {line}")

## agents/team.py — The Team

The team definition has:
- **Team Leader** (Claude Sonnet) — reads your request and delegates
- **3 members**, each a Sonnet agent with specific tools:
  - **Content Writer** — researches topics and writes articles (DataForSEO search + storage)
  - **Image Finder** — finds and inserts images into articles (optional, DataForSEO Images + storage)
  - **AIO Analyzer** — analyzes Google AI Overviews (AIO tools)

The Leader never calls tools directly — it delegates to the right member.

This follows the **project manager pattern**: you talk to one person, they assign the work.

In [ ]:
# Let's look at the team structure
team_path = os.path.abspath("../../output/backend/agents/team.py")
with open(team_path, "r", encoding="utf-8") as f:
    team_code = f.read()

print(f"agents/team.py ({len(team_code.splitlines())} lines)\n")
for i, line in enumerate(team_code.splitlines(), 1):
    print(f"  {i:3d} | {line}")

## agents/ — The AI Workers

The `agents/` directory contains one file per agent:

| File | Agent | Tools | Purpose |
|------|-------|-------|---------|
| `content_writer.py` | Content Writer | DataForSEO search + save_article + list_all_articles | Research and write SEO articles |
| `image_finder.py` | Image Finder | DataForSEO Images + get_article_content + update_article_content | Find and insert images |
| `aio_analyzer.py` | AIO Analyzer | analyze_keyword_aio + optimize_for_aio | Analyze Google AI Overviews |
| `team.py` | Team assembly | (delegates) | Sonnet leader + 3 members |
| `__init__.py` | — | — | Re-exports everything |

**All agents use Claude Sonnet.** The leader and all members are the same model.

**One agent = one file.** Want to change the writer? Open `content_writer.py`. Want to add a fact-checker? Create `fact_checker.py`.

In [ ]:
# Let's look at the Content Writer definition
writer_path = os.path.abspath("../../output/backend/agents/content_writer.py")
with open(writer_path, "r", encoding="utf-8") as f:
    print(f.read())

## tools/ — Capabilities

The `tools/` folder contains what agents **can do**:

| File | Purpose |
|------|---------|
| `storage.py` | Local file storage — save, list, get, update articles |
| `search.py` | Web search via DataForSEO SERP API |
| `aio.py` | AI Overview analysis via DataForSEO + credentials helper |
| `images.py` | Image search via DataForSEO |
| `__init__.py` | Package marker |

`agents/` = **who** (the AI workers with their instructions)

`tools/` = **what** (the capabilities those workers use)

All tools **degrade gracefully**:
- No DataForSEO key → search/images/AIO return empty results, but agents still work
- `get_dataforseo_credentials()` in `aio.py` is the shared credential decoder

## Tracing the Image Flow

When you say "Add images to article on-page-seo":

```
Your request
  |
  v
Team Leader --- decides this is an image task
  |
  v
Image Finder:
  1. get_article_content("on-page-seo")  --- reads the article from storage
  2. search_images("on page SEO")        --- finds images via DataForSEO
  3. Inserts ![alt](url) into the Markdown
  4. update_article_content("on-page-seo", updated_markdown) --- saves changes
  |
  v
content/on-page-seo.md --- now has images!
```

Notice how the Image Finder uses **storage tools** (get + update) plus **image search tools**. It reads the existing article, adds images, and saves the updated version.

## Why This File Structure?

Why not put everything in one file? Here's the philosophy:

**One file = one job.** `content_writer.py` writes articles. `storage.py` saves files. `search.py` searches the web. You never have to guess where something lives.

**One agent = one file.** Want to change the writer's instructions? Open `agents/content_writer.py`. Want to add a fact-checker agent? Create `agents/fact_checker.py` and add it to the team.

**Naming = navigation.** File names tell you what's inside without opening them. If someone says "the article didn't save", you look at `tools/storage.py`.

**`agents/` vs `tools/`** — AI agent definitions vs capabilities. Agents are the "thinking" layer (they use LLMs). Tools are the "doing" layer (they call APIs, read files, save data).

**Flat is better than nested.** 3 members do their jobs directly. No pipeline.py orchestrating steps, no workspace.py bridging layers. Each member has its tools and calls them directly.

## The Complete File Structure

```
output/backend/                 (12 Python files)
├── chat.py                     CLI entry point (~40 lines)
├── serve.py                    Web backend (AgentOS + article routes)
├── agents/                     Agent definitions (who)
│   ├── __init__.py             Re-exports everything
│   ├── content_writer.py       Content Writer (search + storage)
│   ├── image_finder.py         Image Finder (images + storage, optional)
│   ├── aio_analyzer.py         AIO Analyzer (AIO tools)
│   └── team.py                 Agno Team (leader + 3 members)
└── tools/                      Tool definitions (what)
    ├── __init__.py             Package marker
    ├── storage.py              Local file storage (JSON + .md)
    ├── search.py               DataForSEO web search
    ├── aio.py                  AIO analysis + credentials
    └── images.py               DataForSEO image search
```

In [ ]:
# Verify the file structure matches
output_dir = os.path.abspath("../../output/backend")
print("Your production codebase:\n")

for root, dirs, files in os.walk(output_dir):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(output_dir, "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if file.endswith(".py"):
            filepath = os.path.join(root, file)
            with open(filepath, "r", encoding="utf-8") as f:
                lines = len(f.readlines())
            print(f"{sub_indent}{file} ({lines} lines)")

## Summary

Everything connects through **plain Python function calls**. No magic.

- **`serve.py`** — Web backend. AgentOS wraps the team + custom article routes.
- **`chat.py`** — CLI alternative. Same team, terminal interface.
- **`agents/team.py`** — Team Leader (Sonnet) delegates to 3 Sonnet members.
- **`agents/*.py`** — One file per agent. Each has its model, tools, and instructions.
- **`tools/storage.py`** — Local file storage. Articles as `.md` files + JSON metadata.
- **`tools/search.py`** — Web search via DataForSEO.
- **`tools/aio.py`** — AI Overview analysis + shared credentials.
- **`tools/images.py`** — Image search via DataForSEO.

The full flow: **serve.py → agents/team.py → agents/{member}.py → tools/{capability}.py → content/**

The philosophy: **one file = one job, one agent = one file, naming = navigation.**

**Next lesson:** Web fundamentals — understanding client/server, APIs, and streaming before diving into the web interface.

## Exercise

Read-only questions — no API calls needed. You're reading code, the same way Claude Code does when it explores a codebase.

1. Open `output/backend/agents/content_writer.py`. What tools does the Content Writer have? What happens if `DATA_FOR_SEO_API_KEY` is not set?
2. Open `output/backend/agents/image_finder.py`. What does `build_image_finder()` return if DataForSEO is not configured?
3. Trace the import chain: `serve.py` imports from `agents/team.py`, which imports from `agents/content_writer.py`, which imports from `tools/search.py` and `tools/storage.py`. Draw this import graph on paper.
4. Why does `get_dataforseo_credentials()` live in `tools/aio.py` and not in `tools/search.py`? (Hint: who else uses it?)

<details>
<summary>Click to reveal answers</summary>

1. **Content Writer tools:** `DataForSEOSearchTools` (conditional) + `save_article` + `list_all_articles`. If no DataForSEO key, the search tool is simply not added — the writer still works but relies on built-in knowledge.

2. **`build_image_finder()`** returns `None` if DataForSEO is not configured. The team checks `if image_finder is not None` before adding it to members.

3. **Import graph:**
```
serve.py
  └── agents/team.py
        ├── agents/content_writer.py
        │     ├── tools/search.py
        │     ├── tools/storage.py
        │     └── tools/aio.py (credentials)
        ├── agents/image_finder.py
        │     ├── tools/images.py
        │     ├── tools/storage.py
        │     └── tools/aio.py (credentials)
        └── agents/aio_analyzer.py
              └── tools/aio.py
```

4. **`get_dataforseo_credentials()`** lives in `aio.py` because it's shared by `search.py`, `images.py`, and `aio.py` — all three use DataForSEO. Putting it in `aio.py` avoids circular imports since AIO was the first DataForSEO integration.
</details>